In [ ]:
# from pathlib import Path
# from function._01_boundary import get_boundary_file
# from function._02_clipping import clipping_jakarta_boundary
# from function._03_cloud_mask import cloud_masking
# from function._04_rescalling import rescalling_ST_SR
# from function._06_indeks_spektral import index_spectral
# from function._06_outlier import outlier_detection
# from function._07_composit_monthly import composite_image
# from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
# from function._08_statistic_count import count_statistic
# from function._09_labeling import natural_break_labeling
# from function._10_train import train_final_model

# BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
# TIF_PATH = "../../data/raw_data/landsat_30"

# print("1. MEMUAT BATAS ADMINISTRASI")
# print("=" * 60)
# boundary = get_boundary_file(BOUNDARY_PATH)

# print("2. CLIPPING CITRA LANDSAT")
# print("=" * 60)
# images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual = True)

# print("3. CLOUD MASKING")
# print("=" * 60)
# images = cloud_masking(images, is_visual = True)

# print("4. RESCALING BAND")
# print("=" * 60)
# images = rescalling_ST_SR(images, is_visual = True)

# # print("4. RESCALING BAND")
# # print("=" * 60)
# # outlier = outlier_detection(images)

# # print("7. PENGISIAN GAP DENGAN IDW")
# # print("=" * 60)
# # n, r, p, results = test_idw(outlier)

# print("8. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# idw = inverse_distance_weighting(images, 4, 2, 3, is_visual = True)

# print("5. PERHITUNGAN INDEKS SPEKTRAL")
# print("=" * 60)
# images = index_spectral(images)

# print("7. KOMPOSIT BULANAN")
# print("=" * 60)
# images = composite_image(images)

# print("9. PERHITUNGAN STATISTIK KECAMATAN")
# print("=" * 60)
# df = count_statistic(images, boundary)

In [1]:
import json
from pathlib import Path
from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
from function._06_outlier import outlier_detection
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._09_labeling import apply_natural_breaks
from function._10_train import train_final_model

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual = True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual = True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual = True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW")
print("=" * 60)
idw = inverse_distance_weighting(images, 4, 2, 3, is_visual = True)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)
# df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
# pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
model, best_params, metrics, cv_results = train_final_model(df)

# Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
# breaks dari DATA LATIH yang disimpan train_final_model.
with open("models/evaluasi_model.json") as f:
    breaks = json.load(f)["breaks"]

# dropna + reset_index sama persis dengan di train_final_model,
# supaya baris selaras dengan models/prediksi_kecamatan.csv
data_labeled = df.dropna().reset_index(drop = True)
data_labeled["lst_label"] = apply_natural_breaks(data_labeled, breaks)
data_labeled.to_pickle("models/data_labeled.pkl")   # dipakai script analisis

print("\nDistribusi kelas LST (batas dari data latih):")
print(data_labeled["lst_label"].value_counts().sort_index())

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
2019-04 | Gambir | Valid: 684/8,341 (8.20%)
2019-04 | Sawah Besar | Valid: 154/5,941 (2.59%)
2019-04 | Kemayoran | Valid: 616/7,965 (7.73%)
2019-04 | Senen | Valid: 583/4,838 (12.05%)
2019-04 | Menteng | Valid: 346/7,205 (4.80%)
2019-04 | Tanah Abang | Valid: 2,789/11,152 (25.01%)
2019-04 | Johar Baru | Valid: 113/2,627 (4.30%)
2019-04 | Penjaringan | Valid: 2,229/45,005 (4.95%)
2019-04 | Tanjung Priok | Valid: 8,361/25,021 (33.42%)
2019-04 | Koja | Valid: 734/12,958 (5.66%)
2019-04 | Cilincing | Valid: 4,367/48,565 (8.99%)
2019-04 | Pademangan | Valid: 4,375/14,808 (29.54%)
2019-04 | Cengkareng | Valid: 1,853/28,580 (6.48%)
2019-04 | Taman Sari | Valid: 26/5,002 (0.52%)
2019-04 | Kebon Jeruk | Valid: 8,848/19,602 (45.14%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,607/8,188 (31.84%)
2019-04 | Kembangan | Valid: 22,056/27,702 (79.62%)
2019-04 | Setiabudi | Valid: 332/9,914 (3.35%)
2019-04 | Kebayoran Lama | Valid

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/100] Train Loss: 1.0245 Val Loss: 1.0536 Patience: 0/15
Epoch [2/100] Train Loss: 0.9541 Val Loss: 0.9406 Patience: 0/15
Epoch [3/100] Train Loss: 0.8772 Val Loss: 0.8060 Patience: 0/15
Epoch [4/100] Train Loss: 0.7788 Val Loss: 0.7098 Patience: 0/15
Epoch [5/100] Train Loss: 0.7343 Val Loss: 0.7095 Patience: 0/15
Epoch [6/100] Train Loss: 0.7025 Val Loss: 0.6990 Patience: 0/15
Epoch [7/100] Train Loss: 0.7061 Val Loss: 0.6755 Patience: 0/15
Epoch [8/100] Train Loss: 0.6681 Val Loss: 0.6870 Patience: 1/15
Epoch [9/100] Train Loss: 0.6502 Val Loss: 0.6992 Patience: 2/15
Epoch [10/100] Train Loss: 0.6593 Val Loss: 0.7637 Patience: 3/15
Epoch [11/100] Train Loss: 0.6352 Val Loss: 0.6635 Patience: 0/15
Epoch [12/100] Train Loss: 0.6108 Val Loss: 0.6650 Patience: 1/15
Epoch [13/100] Train Loss: 0.6015 Val Loss: 0.6435 Patience: 0/15
Epoch [14/100] Train Loss: 0.6002 Val Loss: 0.6480 Patience: 1/15
Epoch [15/100] Train Loss: 0.5839 Val Loss: 0.6436 Patience: 2/15
Epoch [16/100] Trai

In [ ]:
import json
from pathlib import Path
from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
# from function._06_outlier import outlier_detection
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._09_labeling import apply_natural_breaks
from function._10_train import train_final_model

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual = True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual = True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual = True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW")
print("=" * 60)
idw = inverse_distance_weighting(images, 4, 2, 3, is_visual = True)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)
# df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
# pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
model, best_params, metrics, cv_results = train_final_model(df)

# Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
# breaks dari DATA LATIH yang disimpan train_final_model.
with open("models/evaluasi_model.json") as f:
    breaks = json.load(f)["breaks"]

# dropna + reset_index sama persis dengan di train_final_model,
# supaya baris selaras dengan models/prediksi_kecamatan.csv
data_labeled = df.dropna().reset_index(drop = True)
data_labeled["lst_label"] = apply_natural_breaks(data_labeled, breaks)
data_labeled.to_pickle("models/data_labeled.pkl")   # dipakai script analisis

print("\nDistribusi kelas LST (batas dari data latih):")
print(data_labeled["lst_label"].value_counts().sort_index())

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
2019-04 | Gambir | Valid: 684/8,341 (8.20%)
2019-04 | Sawah Besar | Valid: 154/5,941 (2.59%)
2019-04 | Kemayoran | Valid: 616/7,965 (7.73%)
2019-04 | Senen | Valid: 583/4,838 (12.05%)
2019-04 | Menteng | Valid: 346/7,205 (4.80%)
2019-04 | Tanah Abang | Valid: 2,789/11,152 (25.01%)
2019-04 | Johar Baru | Valid: 113/2,627 (4.30%)
2019-04 | Penjaringan | Valid: 2,229/45,005 (4.95%)
2019-04 | Tanjung Priok | Valid: 8,361/25,021 (33.42%)
2019-04 | Koja | Valid: 734/12,958 (5.66%)
2019-04 | Cilincing | Valid: 4,367/48,565 (8.99%)
2019-04 | Pademangan | Valid: 4,375/14,808 (29.54%)
2019-04 | Cengkareng | Valid: 1,853/28,580 (6.48%)
2019-04 | Taman Sari | Valid: 26/5,002 (0.52%)
2019-04 | Kebon Jeruk | Valid: 8,848/19,602 (45.14%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,607/8,188 (31.84%)
2019-04 | Kembangan | Valid: 22,056/27,702 (79.62%)
2019-04 | Setiabudi | Valid: 332/9,914 (3.35%)
2019-04 | Kebayoran Lama | Valid

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/100] Train Loss: 1.0532 Val Loss: 1.0656 Patience: 0/15
Epoch [2/100] Train Loss: 0.9503 Val Loss: 0.8513 Patience: 0/15
Epoch [3/100] Train Loss: 0.8271 Val Loss: 0.6853 Patience: 0/15
Epoch [4/100] Train Loss: 0.7489 Val Loss: 0.6535 Patience: 0/15
Epoch [5/100] Train Loss: 0.7212 Val Loss: 0.6266 Patience: 0/15
Epoch [6/100] Train Loss: 0.6979 Val Loss: 0.6099 Patience: 0/15
Epoch [7/100] Train Loss: 0.6673 Val Loss: 0.6194 Patience: 1/15
Epoch [8/100] Train Loss: 0.6740 Val Loss: 0.6167 Patience: 2/15
Epoch [9/100] Train Loss: 0.6450 Val Loss: 0.5878 Patience: 0/15
Epoch [10/100] Train Loss: 0.6515 Val Loss: 0.5931 Patience: 1/15
Epoch [11/100] Train Loss: 0.6307 Val Loss: 0.5988 Patience: 2/15
Epoch [12/100] Train Loss: 0.6519 Val Loss: 0.5804 Patience: 0/15
Epoch [13/100] Train Loss: 0.6342 Val Loss: 0.5657 Patience: 0/15
Epoch [14/100] Train Loss: 0.5902 Val Loss: 0.5557 Patience: 0/15
Epoch [15/100] Train Loss: 0.6101 Val Loss: 0.6058 Patience: 1/15
Epoch [16/100] Trai

In [1]:
import json
from pathlib import Path
from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._09_labeling import apply_natural_breaks
from function._10_train import train_final_model

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual = True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual = True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual = True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)")
print("=" * 60)
images = inverse_distance_weighting(
    images, boundary, 4, 2, 3,
    report_path = "output/idw_report.csv"
)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)
# df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
# pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
model, best_params, metrics, cv_results = train_final_model(df)

# Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
# breaks dari DATA LATIH yang disimpan train_final_model.
with open("models/evaluasi_model.json") as f:
    breaks = json.load(f)["breaks"]

# dropna + reset_index sama persis dengan di train_final_model,
# supaya baris selaras dengan models/prediksi_kecamatan.csv
data_labeled = df.dropna().reset_index(drop = True)
data_labeled["lst_label"] = apply_natural_breaks(data_labeled, breaks)
data_labeled.to_pickle("models/data_labeled.pkl")   # dipakai script analisis

print("\nDistribusi kelas LST (batas dari data latih):")
print(data_labeled["lst_label"].value_counts().sort_index())

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
2019-04 | Gambir | Valid: 681/8,341 (8.16%)
2019-04 | Sawah Besar | Valid: 145/5,941 (2.44%)
2019-04 | Kemayoran | Valid: 612/7,965 (7.68%)
2019-04 | Senen | Valid: 578/4,838 (11.95%)
2019-04 | Menteng | Valid: 345/7,205 (4.79%)
2019-04 | Tanah Abang | Valid: 2,771/11,152 (24.85%)
2019-04 | Johar Baru | Valid: 110/2,627 (4.19%)
2019-04 | Penjaringan | Valid: 2,227/45,005 (4.95%)
2019-04 | Tanjung Priok | Valid: 8,360/25,021 (33.41%)
2019-04 | Koja | Valid: 734/12,958 (5.66%)
2019-04 | Cilincing | Valid: 4,367/48,565 (8.99%)
2019-04 | Pademangan | Valid: 4,370/14,808 (29.51%)
2019-04 | Cengkareng | Valid: 1,848/28,580 (6.47%)
2019-04 | Taman Sari | Valid: 26/5,002 (0.52%)
2019-04 | Kebon Jeruk | Valid: 8,837/19,602 (45.08%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,605/8,188 (31.81%)
2019-04 | Kembangan | Valid: 22,032/27,702 (79.53%)
2019-04 | Setiabudi | Valid: 330/9,914 (3.33%)
2019-04 | Kebayoran Lama | Valid

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/100] Train Loss: 1.0246 Val Loss: 1.0542 Patience: 0/15
Epoch [2/100] Train Loss: 0.9539 Val Loss: 0.9241 Patience: 0/15
Epoch [3/100] Train Loss: 0.8776 Val Loss: 0.7870 Patience: 0/15
Epoch [4/100] Train Loss: 0.7785 Val Loss: 0.6957 Patience: 0/15
Epoch [5/100] Train Loss: 0.7326 Val Loss: 0.7079 Patience: 1/15
Epoch [6/100] Train Loss: 0.7001 Val Loss: 0.6750 Patience: 0/15
Epoch [7/100] Train Loss: 0.6989 Val Loss: 0.6536 Patience: 0/15
Epoch [8/100] Train Loss: 0.6698 Val Loss: 0.6610 Patience: 1/15
Epoch [9/100] Train Loss: 0.6517 Val Loss: 0.6731 Patience: 2/15
Epoch [10/100] Train Loss: 0.6579 Val Loss: 0.7339 Patience: 3/15
Epoch [11/100] Train Loss: 0.6351 Val Loss: 0.6454 Patience: 0/15
Epoch [12/100] Train Loss: 0.6075 Val Loss: 0.6386 Patience: 0/15
Epoch [13/100] Train Loss: 0.6004 Val Loss: 0.6313 Patience: 0/15
Epoch [14/100] Train Loss: 0.6002 Val Loss: 0.6188 Patience: 0/15
Epoch [15/100] Train Loss: 0.5851 Val Loss: 0.6203 Patience: 1/15
Epoch [16/100] Trai

In [ ]:
import numpy as np
import pandas as pd

# 1. Fungsi menghitung rata-rata LST dengan mengabaikan NoData
def rata_rata_valid(nilai):
    nilai = np.array(nilai, dtype=float)
    nilai[nilai == -9999] = np.nan
    return np.nanmean(nilai)


# 2. Menghitung rata-rata LST setiap kecamatan dan periode
data_labeled["lst_mean"] = data_labeled["lst"].apply(rata_rata_valid)

lst_kecamatan = (
    data_labeled.groupby(["period", "subdistrict"])["lst_mean"]
    .mean()
    .reset_index(name="lst_kecamatan")
)


# 3. Mengambil label kelas setiap kecamatan dan periode
kelas_kecamatan = (
    data_labeled.groupby(["period", "subdistrict"])["lst_label"]
    .first()
    .reset_index()
)


# 4. Menggabungkan nilai LST dan label kelas
hasil = lst_kecamatan.merge(
    kelas_kecamatan,
    on=["period", "subdistrict"]
)


# 5. Mengubah label numerik menjadi nama kelas
nama_kelas = {
    0: "Rendah",
    1: "Normal",
    2: "Tinggi"
}

hasil["kelas"] = hasil["lst_label"].map(nama_kelas)


# 6. Menghitung rata-rata LST seluruh Jakarta setiap periode
rata2_jakarta = (
    hasil.groupby("period")["lst_kecamatan"]
    .mean()
    .rename("rata2_jakarta")
)

hasil = hasil.merge(
    rata2_jakarta,
    on="period"
)


# 7. Menghitung selisih LST kecamatan terhadap rata-rata Jakarta
hasil["selisih_jakarta"] = (
    hasil["lst_kecamatan"] - hasil["rata2_jakarta"]
)


# 8. Menghitung statistik selisih berdasarkan kelas
rata_kelas = (
    hasil.groupby("kelas")["selisih_jakarta"]
    .agg(["count", "mean", "std", "min", "max"])
)

print("Rata-rata Selisih LST Berdasarkan Kelas:")
print(rata_kelas.round(3))


# 9. Menyimpan hasil ke CSV
hasil.round(3).to_csv("selisih_lst.csv", index=False)

print("\nData berhasil disimpan ke selisih_lst.csv")

In [ ]:
import json
from pathlib import Path
from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._09_labeling import apply_natural_breaks
from function._10_train import train_final_model

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual = True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual = True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual = True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)")
print("=" * 60)
images = inverse_distance_weighting(
    images, boundary, 4, 2, 3,
    report_path = "output/idw_report.csv"
)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)

# df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
# pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
model, best_params, metrics, cv_results = train_final_model(df)

# Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
# breaks dari DATA LATIH yang disimpan train_final_model.
with open("models/evaluasi_model.json") as f:
    breaks = json.load(f)["breaks"]

# dropna + reset_index sama persis dengan di train_final_model,
# supaya baris selaras dengan models/prediksi_kecamatan.csv
data_labeled = df.dropna().reset_index(drop = True)
data_labeled["lst_label"] = apply_natural_breaks(data_labeled, breaks)
data_labeled.to_pickle("models/data_labeled.pkl")   # dipakai script analisis

print("\nDistribusi kelas LST (batas dari data latih):")
print(data_labeled["lst_label"].value_counts().sort_index())

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
2019-04 | Gambir | Valid: 681/8,341 (8.16%)
2019-04 | Sawah Besar | Valid: 145/5,941 (2.44%)
2019-04 | Kemayoran | Valid: 612/7,965 (7.68%)
2019-04 | Senen | Valid: 578/4,838 (11.95%)
2019-04 | Menteng | Valid: 345/7,205 (4.79%)
2019-04 | Tanah Abang | Valid: 2,771/11,152 (24.85%)
2019-04 | Johar Baru | Valid: 110/2,627 (4.19%)
2019-04 | Penjaringan | Valid: 2,227/45,005 (4.95%)
2019-04 | Tanjung Priok | Valid: 8,360/25,021 (33.41%)
2019-04 | Koja | Valid: 734/12,958 (5.66%)
2019-04 | Cilincing | Valid: 4,367/48,565 (8.99%)
2019-04 | Pademangan | Valid: 4,370/14,808 (29.51%)
2019-04 | Cengkareng | Valid: 1,848/28,580 (6.47%)
2019-04 | Taman Sari | Valid: 26/5,002 (0.52%)
2019-04 | Kebon Jeruk | Valid: 8,837/19,602 (45.08%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,605/8,188 (31.81%)
2019-04 | Kembangan | Valid: 22,032/27,702 (79.53%)
2019-04 | Setiabudi | Valid: 330/9,914 (3.33%)
2019-04 | Kebayoran Lama | Valid

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/100] Train Loss: 1.0168 Val Loss: 1.0408 Patience: 0/15
Epoch [2/100] Train Loss: 0.9402 Val Loss: 0.8869 Patience: 0/15
Epoch [3/100] Train Loss: 0.7673 Val Loss: 0.8014 Patience: 0/15
Epoch [4/100] Train Loss: 0.7063 Val Loss: 0.7595 Patience: 0/15
Epoch [5/100] Train Loss: 0.6539 Val Loss: 0.7417 Patience: 0/15
Epoch [6/100] Train Loss: 0.6482 Val Loss: 0.7859 Patience: 1/15
Epoch [7/100] Train Loss: 0.6418 Val Loss: 0.7924 Patience: 2/15
Epoch [8/100] Train Loss: 0.6597 Val Loss: 0.8625 Patience: 3/15
Epoch [9/100] Train Loss: 0.6325 Val Loss: 0.8093 Patience: 4/15
Epoch [10/100] Train Loss: 0.6216 Val Loss: 0.7510 Patience: 5/15
Epoch [11/100] Train Loss: 0.6074 Val Loss: 0.7646 Patience: 6/15
Epoch [12/100] Train Loss: 0.6255 Val Loss: 0.6998 Patience: 0/15
Epoch [13/100] Train Loss: 0.5991 Val Loss: 0.7041 Patience: 1/15
Epoch [14/100] Train Loss: 0.5888 Val Loss: 0.6725 Patience: 0/15
Epoch [15/100] Train Loss: 0.5885 Val Loss: 0.7307 Patience: 1/15
Epoch [16/100] Trai

In [1]:
import json
from pathlib import Path
from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._09_labeling import apply_natural_breaks
from function._10_train import train_final_model

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual = True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual = True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual = True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)")
print("=" * 60)
images = inverse_distance_weighting(
    images, boundary, 4, 2, 3,
    report_path = "output/idw_report.csv"
)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)

# df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
# pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
model, best_params, metrics, cv_results = train_final_model(df)

# Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
# breaks dari DATA LATIH yang disimpan train_final_model.
with open("models/evaluasi_model.json") as f:
    breaks = json.load(f)["breaks"]

# dropna + reset_index sama persis dengan di train_final_model,
# supaya baris selaras dengan models/prediksi_kecamatan.csv
data_labeled = df.dropna().reset_index(drop = True)
data_labeled["lst_label"] = apply_natural_breaks(data_labeled, breaks)
data_labeled.to_pickle("models/data_labeled.pkl")   # dipakai script analisis

print("\nDistribusi kelas LST (batas dari data latih):")
print(data_labeled["lst_label"].value_counts().sort_index())

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
2019-04 | Gambir | Valid: 681/8,341 (8.16%)
2019-04 | Sawah Besar | Valid: 145/5,941 (2.44%)
2019-04 | Kemayoran | Valid: 612/7,965 (7.68%)
2019-04 | Senen | Valid: 578/4,838 (11.95%)
2019-04 | Menteng | Valid: 345/7,205 (4.79%)
2019-04 | Tanah Abang | Valid: 2,771/11,152 (24.85%)
2019-04 | Johar Baru | Valid: 110/2,627 (4.19%)
2019-04 | Penjaringan | Valid: 2,227/45,005 (4.95%)
2019-04 | Tanjung Priok | Valid: 8,360/25,021 (33.41%)
2019-04 | Koja | Valid: 734/12,958 (5.66%)
2019-04 | Cilincing | Valid: 4,367/48,565 (8.99%)
2019-04 | Pademangan | Valid: 4,370/14,808 (29.51%)
2019-04 | Cengkareng | Valid: 1,848/28,580 (6.47%)
2019-04 | Taman Sari | Valid: 26/5,002 (0.52%)
2019-04 | Kebon Jeruk | Valid: 8,837/19,602 (45.08%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,605/8,188 (31.81%)
2019-04 | Kembangan | Valid: 22,032/27,702 (79.53%)
2019-04 | Setiabudi | Valid: 330/9,914 (3.33%)
2019-04 | Kebayoran Lama | Valid

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/100] Train Loss: 1.0510 Val Loss: 0.9825 Patience: 0/15
Epoch [2/100] Train Loss: 0.9659 Val Loss: 0.8771 Patience: 0/15
Epoch [3/100] Train Loss: 0.8073 Val Loss: 0.6074 Patience: 0/15
Epoch [4/100] Train Loss: 0.7557 Val Loss: 0.7030 Patience: 1/15
Epoch [5/100] Train Loss: 0.7058 Val Loss: 0.5750 Patience: 0/15
Epoch [6/100] Train Loss: 0.6747 Val Loss: 0.5240 Patience: 0/15
Epoch [7/100] Train Loss: 0.6700 Val Loss: 0.5521 Patience: 1/15
Epoch [8/100] Train Loss: 0.6998 Val Loss: 0.5366 Patience: 2/15
Epoch [9/100] Train Loss: 0.6444 Val Loss: 0.4948 Patience: 0/15
Epoch [10/100] Train Loss: 0.6527 Val Loss: 0.5076 Patience: 1/15
Epoch [11/100] Train Loss: 0.6489 Val Loss: 0.5339 Patience: 2/15
Epoch [12/100] Train Loss: 0.6269 Val Loss: 0.4954 Patience: 3/15
Epoch [13/100] Train Loss: 0.6108 Val Loss: 0.4796 Patience: 0/15
Epoch [14/100] Train Loss: 0.6203 Val Loss: 0.5066 Patience: 1/15
Epoch [15/100] Train Loss: 0.5973 Val Loss: 0.4786 Patience: 0/15
Epoch [16/100] Trai

In [1]:
import json
from pathlib import Path
from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._09_labeling import apply_natural_breaks
from function._10_train import train_final_model

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual = True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual = True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual = True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)")
print("=" * 60)
images = inverse_distance_weighting(
    images, boundary, 4, 2, 3,
    report_path = "output/idw_report.csv"
)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)

# df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
# pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
model, best_params, metrics, cv_results = train_final_model(df)

# Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
# breaks dari DATA LATIH yang disimpan train_final_model.
with open("models/evaluasi_model.json") as f:
    breaks = json.load(f)["breaks"]

# dropna + reset_index sama persis dengan di train_final_model,
# supaya baris selaras dengan models/prediksi_kecamatan.csv
data_labeled = df.dropna().reset_index(drop = True)
data_labeled["lst_label"] = apply_natural_breaks(data_labeled, breaks)
data_labeled.to_pickle("models/data_labeled.pkl")   # dipakai script analisis

print("\nDistribusi kelas LST (batas dari data latih):")
print(data_labeled["lst_label"].value_counts().sort_index())

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
2019-04 | Gambir | Valid: 681/8,341 (8.16%)
2019-04 | Sawah Besar | Valid: 145/5,941 (2.44%)
2019-04 | Kemayoran | Valid: 612/7,965 (7.68%)
2019-04 | Senen | Valid: 578/4,838 (11.95%)
2019-04 | Menteng | Valid: 345/7,205 (4.79%)
2019-04 | Tanah Abang | Valid: 2,771/11,152 (24.85%)
2019-04 | Johar Baru | Valid: 110/2,627 (4.19%)
2019-04 | Penjaringan | Valid: 2,227/45,005 (4.95%)
2019-04 | Tanjung Priok | Valid: 8,360/25,021 (33.41%)
2019-04 | Koja | Valid: 734/12,958 (5.66%)
2019-04 | Cilincing | Valid: 4,367/48,565 (8.99%)
2019-04 | Pademangan | Valid: 4,370/14,808 (29.51%)
2019-04 | Cengkareng | Valid: 1,848/28,580 (6.47%)
2019-04 | Taman Sari | Valid: 26/5,002 (0.52%)
2019-04 | Kebon Jeruk | Valid: 8,837/19,602 (45.08%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,605/8,188 (31.81%)
2019-04 | Kembangan | Valid: 22,032/27,702 (79.53%)
2019-04 | Setiabudi | Valid: 330/9,914 (3.33%)
2019-04 | Kebayoran Lama | Valid

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/200] Train Loss: 1.0510 Val Loss: 0.9825 Patience: 0/30
Epoch [2/200] Train Loss: 0.9659 Val Loss: 0.8771 Patience: 0/30
Epoch [3/200] Train Loss: 0.8073 Val Loss: 0.6074 Patience: 0/30
Epoch [4/200] Train Loss: 0.7557 Val Loss: 0.7030 Patience: 1/30
Epoch [5/200] Train Loss: 0.7058 Val Loss: 0.5750 Patience: 0/30
Epoch [6/200] Train Loss: 0.6747 Val Loss: 0.5240 Patience: 0/30
Epoch [7/200] Train Loss: 0.6700 Val Loss: 0.5521 Patience: 1/30
Epoch [8/200] Train Loss: 0.6998 Val Loss: 0.5366 Patience: 2/30
Epoch [9/200] Train Loss: 0.6444 Val Loss: 0.4948 Patience: 0/30
Epoch [10/200] Train Loss: 0.6527 Val Loss: 0.5076 Patience: 1/30
Epoch [11/200] Train Loss: 0.6489 Val Loss: 0.5339 Patience: 2/30
Epoch [12/200] Train Loss: 0.6269 Val Loss: 0.4954 Patience: 3/30
Epoch [13/200] Train Loss: 0.6108 Val Loss: 0.4796 Patience: 0/30
Epoch [14/200] Train Loss: 0.6203 Val Loss: 0.5066 Patience: 1/30
Epoch [15/200] Train Loss: 0.5973 Val Loss: 0.4786 Patience: 0/30
Epoch [16/200] Trai

In [ ]:

import json
from pathlib import Path

from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._09_labeling import apply_natural_breaks
from function._10_train import train_final_model, jalankan

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual=True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual=True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual=True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)")
print("=" * 60)
images = inverse_distance_weighting(
    images, boundary, 4, 2, 3,
    report_path="output/idw_report.csv"
)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)

# df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
# pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
model, best_params, metrics, cv_results = train_final_model(df)

# Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
# breaks dari DATA LATIH yang disimpan train_final_model.
with open("models/evaluasi_model.json") as f:
    breaks = json.load(f)["breaks"]

# dropna + reset_index sama persis dengan di train_final_model,
# supaya baris selaras dengan models/prediksi_kecamatan.csv
data_labeled = df.dropna().reset_index(drop=True)
data_labeled["lst_label"] = apply_natural_breaks(data_labeled, breaks)
data_labeled.to_pickle("models/data_labeled.pkl")  # dipakai script analisis

print("\nDistribusi kelas LST (batas dari data latih):")
print(data_labeled["lst_label"].value_counts().sort_index())


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
2019-04 | Gambir | Valid: 681/8,341 (8.16%)
2019-04 | Sawah Besar | Valid: 145/5,941 (2.44%)
2019-04 | Kemayoran | Valid: 612/7,965 (7.68%)
2019-04 | Senen | Valid: 578/4,838 (11.95%)
2019-04 | Menteng | Valid: 345/7,205 (4.79%)
2019-04 | Tanah Abang | Valid: 2,771/11,152 (24.85%)
2019-04 | Johar Baru | Valid: 110/2,627 (4.19%)
2019-04 | Penjaringan | Valid: 2,227/45,005 (4.95%)
2019-04 | Tanjung Priok | Valid: 8,360/25,021 (33.41%)
2019-04 | Koja | Valid: 734/12,958 (5.66%)
2019-04 | Cilincing | Valid: 4,367/48,565 (8.99%)
2019-04 | Pademangan | Valid: 4,370/14,808 (29.51%)
2019-04 | Cengkareng | Valid: 1,848/28,580 (6.47%)
2019-04 | Taman Sari | Valid: 26/5,002 (0.52%)
2019-04 | Kebon Jeruk | Valid: 8,837/19,602 (45.08%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,605/8,188 (31.81%)
2019-04 | Kembangan | Valid: 22,032/27,702 (79.53%)
2019-04 | Setiabudi | Valid: 330/9,914 (3.33%)
2019-04 | Kebayoran Lama | Valid

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/200] Train Loss: 1.0510 Val Loss: 0.9825 Patience: 0/30
Epoch [2/200] Train Loss: 0.9659 Val Loss: 0.8771 Patience: 0/30
Epoch [3/200] Train Loss: 0.8073 Val Loss: 0.6074 Patience: 0/30
Epoch [4/200] Train Loss: 0.7557 Val Loss: 0.7030 Patience: 1/30
Epoch [5/200] Train Loss: 0.7058 Val Loss: 0.5750 Patience: 0/30
Epoch [6/200] Train Loss: 0.6747 Val Loss: 0.5240 Patience: 0/30
Epoch [7/200] Train Loss: 0.6700 Val Loss: 0.5521 Patience: 1/30
Epoch [8/200] Train Loss: 0.6998 Val Loss: 0.5366 Patience: 2/30
Epoch [9/200] Train Loss: 0.6444 Val Loss: 0.4948 Patience: 0/30
Epoch [10/200] Train Loss: 0.6527 Val Loss: 0.5076 Patience: 1/30
Epoch [11/200] Train Loss: 0.6489 Val Loss: 0.5339 Patience: 2/30
Epoch [12/200] Train Loss: 0.6269 Val Loss: 0.4954 Patience: 3/30
Epoch [13/200] Train Loss: 0.6108 Val Loss: 0.4796 Patience: 0/30
Epoch [14/200] Train Loss: 0.6203 Val Loss: 0.5066 Patience: 1/30
Epoch [15/200] Train Loss: 0.5973 Val Loss: 0.4786 Patience: 0/30
Epoch [16/200] Trai

In [1]:
import json
from pathlib import Path

from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._10_train import train_final_model, apply_labels

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

# [DITAMBAH] daftar skenario pelabelan yang dilatih berurutan:
# "global" | "normalisasi" | "per_periode"
MODES = ["global", "normalisasi", "per_periode"]

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual=True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual=True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual=True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)")
print("=" * 60)
images = inverse_distance_weighting(
    images, boundary, 4, 2, 3,
    report_path="output/idw_report.csv"
)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)

# [DIUBAH] loop: satu model dilatih untuk setiap skenario pelabelan.
# Bagian 1-9 hanya dijalankan sekali; df dipakai bersama oleh ketiga skenario.
for MODE in MODES:
    SUFFIX = "" if MODE == "global" else f"_{MODE}"

    print("\n" + "=" * 60)
    print(f"SKENARIO: {MODE}")
    print("=" * 60)

    # df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
    # pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
    model, best_params, metrics, cv_results = train_final_model(df, label_mode=MODE)

    # Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
    # breaks dari DATA LATIH yang disimpan train_final_model.
    with open(f"models/evaluasi_model{SUFFIX}.json") as f:
        breaks = json.load(f)["breaks"]

    # dropna + reset_index sama persis dengan di train_final_model,
    # supaya baris selaras dengan models/prediksi_kecamatan{SUFFIX}.csv
    data_labeled = df.dropna().reset_index(drop=True)
    data_labeled["lst_label"] = apply_labels(data_labeled, breaks)
    data_labeled.to_pickle(f"models/data_labeled{SUFFIX}.pkl")  # dipakai script analisis

    print(f"\nDistribusi kelas LST ({MODE}, batas dari data latih):")
    print(data_labeled["lst_label"].value_counts().sort_index())

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
2019-04 | Gambir | Valid: 681/8,341 (8.16%)
2019-04 | Sawah Besar | Valid: 145/5,941 (2.44%)
2019-04 | Kemayoran | Valid: 612/7,965 (7.68%)
2019-04 | Senen | Valid: 578/4,838 (11.95%)
2019-04 | Menteng | Valid: 345/7,205 (4.79%)
2019-04 | Tanah Abang | Valid: 2,771/11,152 (24.85%)
2019-04 | Johar Baru | Valid: 110/2,627 (4.19%)
2019-04 | Penjaringan | Valid: 2,227/45,005 (4.95%)
2019-04 | Tanjung Priok | Valid: 8,360/25,021 (33.41%)
2019-04 | Koja | Valid: 734/12,958 (5.66%)
2019-04 | Cilincing | Valid: 4,367/48,565 (8.99%)
2019-04 | Pademangan | Valid: 4,370/14,808 (29.51%)
2019-04 | Cengkareng | Valid: 1,848/28,580 (6.47%)
2019-04 | Taman Sari | Valid: 26/5,002 (0.52%)
2019-04 | Kebon Jeruk | Valid: 8,837/19,602 (45.08%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,605/8,188 (31.81%)
2019-04 | Kembangan | Valid: 22,032/27,702 (79.53%)
2019-04 | Setiabudi | Valid: 330/9,914 (3.33%)
2019-04 | Kebayoran Lama | Valid

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/200] Train Loss: 1.0510 Val Loss: 0.9825 Patience: 0/30
Epoch [2/200] Train Loss: 0.9659 Val Loss: 0.8771 Patience: 0/30
Epoch [3/200] Train Loss: 0.8073 Val Loss: 0.6074 Patience: 0/30
Epoch [4/200] Train Loss: 0.7557 Val Loss: 0.7030 Patience: 1/30
Epoch [5/200] Train Loss: 0.7058 Val Loss: 0.5750 Patience: 0/30
Epoch [6/200] Train Loss: 0.6747 Val Loss: 0.5240 Patience: 0/30
Epoch [7/200] Train Loss: 0.6700 Val Loss: 0.5521 Patience: 1/30
Epoch [8/200] Train Loss: 0.6998 Val Loss: 0.5366 Patience: 2/30
Epoch [9/200] Train Loss: 0.6444 Val Loss: 0.4948 Patience: 0/30
Epoch [10/200] Train Loss: 0.6527 Val Loss: 0.5076 Patience: 1/30
Epoch [11/200] Train Loss: 0.6489 Val Loss: 0.5339 Patience: 2/30
Epoch [12/200] Train Loss: 0.6269 Val Loss: 0.4954 Patience: 3/30
Epoch [13/200] Train Loss: 0.6108 Val Loss: 0.4796 Patience: 0/30
Epoch [14/200] Train Loss: 0.6203 Val Loss: 0.5066 Patience: 1/30
Epoch [15/200] Train Loss: 0.5973 Val Loss: 0.4786 Patience: 0/30
Epoch [16/200] Trai

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/200] Train Loss: 0.9533 Val Loss: 0.8616 Patience: 0/30
Epoch [2/200] Train Loss: 0.9185 Val Loss: 0.9021 Patience: 1/30
Epoch [3/200] Train Loss: 0.8978 Val Loss: 0.8329 Patience: 0/30
Epoch [4/200] Train Loss: 0.8361 Val Loss: 0.7973 Patience: 0/30
Epoch [5/200] Train Loss: 0.8022 Val Loss: 0.8326 Patience: 1/30
Epoch [6/200] Train Loss: 0.7919 Val Loss: 0.7855 Patience: 0/30
Epoch [7/200] Train Loss: 0.7720 Val Loss: 0.7586 Patience: 0/30
Epoch [8/200] Train Loss: 0.7404 Val Loss: 0.7738 Patience: 1/30
Epoch [9/200] Train Loss: 0.7300 Val Loss: 0.7912 Patience: 2/30
Epoch [10/200] Train Loss: 0.7117 Val Loss: 0.8185 Patience: 3/30
Epoch [11/200] Train Loss: 0.7315 Val Loss: 0.7922 Patience: 4/30
Epoch [12/200] Train Loss: 0.6839 Val Loss: 0.8304 Patience: 5/30
Epoch [13/200] Train Loss: 0.6890 Val Loss: 0.8136 Patience: 6/30
Epoch [14/200] Train Loss: 0.6657 Val Loss: 0.8295 Patience: 7/30
Epoch [15/200] Train Loss: 0.6592 Val Loss: 0.8563 Patience: 8/30
Epoch [16/200] Trai

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/200] Train Loss: 1.0210 Val Loss: 0.8855 Patience: 0/30
Epoch [2/200] Train Loss: 0.9815 Val Loss: 0.9372 Patience: 1/30
Epoch [3/200] Train Loss: 0.9667 Val Loss: 0.8470 Patience: 0/30
Epoch [4/200] Train Loss: 0.9212 Val Loss: 0.8082 Patience: 0/30
Epoch [5/200] Train Loss: 0.8715 Val Loss: 0.7804 Patience: 0/30
Epoch [6/200] Train Loss: 0.8472 Val Loss: 0.7549 Patience: 0/30
Epoch [7/200] Train Loss: 0.8347 Val Loss: 0.7370 Patience: 0/30
Epoch [8/200] Train Loss: 0.8178 Val Loss: 0.7833 Patience: 1/30
Epoch [9/200] Train Loss: 0.8181 Val Loss: 0.7606 Patience: 2/30
Epoch [10/200] Train Loss: 0.8044 Val Loss: 0.7591 Patience: 3/30
Epoch [11/200] Train Loss: 0.7970 Val Loss: 0.7482 Patience: 4/30
Epoch [12/200] Train Loss: 0.7800 Val Loss: 0.8143 Patience: 5/30
Epoch [13/200] Train Loss: 0.7634 Val Loss: 0.7624 Patience: 6/30
Epoch [14/200] Train Loss: 0.7558 Val Loss: 0.7638 Patience: 7/30
Epoch [15/200] Train Loss: 0.7598 Val Loss: 0.7801 Patience: 8/30
Epoch [16/200] Trai

In [1]:
import json
from pathlib import Path

from function._01_boundary import get_boundary_file
from function._02_clipping import clipping_jakarta_boundary
from function._03_cloud_mask import cloud_masking
from function._04_rescalling import rescalling_ST_SR
from function._06_indeks_spektral import index_spectral
from function._07_composit_monthly import composite_image
from function._05_inverse_distance_weighting import inverse_distance_weighting, test_idw
from function._08_statistic_count import count_statistic
from function._10_train import train_final_model, apply_labels

BOUNDARY_PATH = "../../data/raw_data/boundary.gpkg"
TIF_PATH = "../../data/raw_data/landsat_30"

# [DITAMBAH] daftar skenario pelabelan yang dilatih berurutan:
# "global" | "normalisasi" | "per_periode"
MODES = ["per_periode"]

print("1. MEMUAT BATAS ADMINISTRASI")
print("=" * 60)
boundary = get_boundary_file(BOUNDARY_PATH)

print("2. CLIPPING CITRA LANDSAT")
print("=" * 60)
images = clipping_jakarta_boundary(boundary, TIF_PATH, is_visual=True)

print("3. CLOUD MASKING")
print("=" * 60)
images = cloud_masking(images, is_visual=True)

print("4. RESCALING BAND")
print("=" * 60)
images = rescalling_ST_SR(images, is_visual=True)

# print("4. RESCALING BAND")
# print("=" * 60)
# outlier = outlier_detection(images)

# print("7. PENGISIAN GAP DENGAN IDW")
# print("=" * 60)
# n, r, p, results = test_idw(outlier)

print("8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)")
print("=" * 60)
images = inverse_distance_weighting(
    images, boundary, 4, 2, 3,
    report_path="output/idw_report.csv"
)

print("5. PERHITUNGAN INDEKS SPEKTRAL")
print("=" * 60)
images = index_spectral(images)

print("7. KOMPOSIT BULANAN")
print("=" * 60)
images = composite_image(images)

print("9. PERHITUNGAN STATISTIK KECAMATAN")
print("=" * 60)
df = count_statistic(images, boundary)

print("10. PELATIHAN MODEL")
print("=" * 60)

# [DIUBAH] loop: satu model dilatih untuk setiap skenario pelabelan.
# Bagian 1-9 hanya dijalankan sekali; df dipakai bersama oleh ketiga skenario.
for MODE in MODES:
    SUFFIX = "" if MODE == "global" else f"_{MODE}"

    print("\n" + "=" * 60)
    print(f"SKENARIO: {MODE}")
    print("=" * 60)

    # df BELUM berlabel. Pemisahan data uji, batas natural breaks per fold,
    # pencarian parameter, dan model final semuanya diurus di dalam train_final_model.
    model, best_params, metrics, cv_results = train_final_model(df, label_mode=MODE)

    # Label untuk analisis deskriptif (bukan evaluasi model): pakai batas natural
    # breaks dari DATA LATIH yang disimpan train_final_model.
    with open(f"models/evaluasi_model{SUFFIX}.json") as f:
        breaks = json.load(f)["breaks"]

    # dropna + reset_index sama persis dengan di train_final_model,
    # supaya baris selaras dengan models/prediksi_kecamatan{SUFFIX}.csv
    data_labeled = df.dropna().reset_index(drop=True)
    data_labeled["lst_label"] = apply_labels(data_labeled, breaks)
    data_labeled.to_pickle(f"models/data_labeled{SUFFIX}.pkl")  # dipakai script analisis

    print(f"\nDistribusi kelas LST ({MODE}, batas dari data latih):")
    print(data_labeled["lst_label"].value_counts().sort_index())

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


1. MEMUAT BATAS ADMINISTRASI
2. CLIPPING CITRA LANDSAT
3. CLOUD MASKING
4. RESCALING BAND
8. PENGISIAN GAP DENGAN IDW (PER KECAMATAN)
5. PERHITUNGAN INDEKS SPEKTRAL
7. KOMPOSIT BULANAN


c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_07_composit_monthly.py:19: RuntimeWarning: All-NaN slice encountered
  median_image = np.nanmedian(image_list, axis = 0)


9. PERHITUNGAN STATISTIK KECAMATAN
DIKECUALIKAN: Gambir | 2019-04
DIKECUALIKAN: Sawah Besar | 2019-04
DIKECUALIKAN: Kemayoran | 2019-04
2019-04 | Senen | Valid: 578/4,838 (11.95%)
DIKECUALIKAN: Menteng | 2019-04
2019-04 | Tanah Abang | Valid: 2,771/11,152 (24.85%)
DIKECUALIKAN: Johar Baru | 2019-04
DIKECUALIKAN: Penjaringan | 2019-04
2019-04 | Tanjung Priok | Valid: 8,360/25,021 (33.41%)
DIKECUALIKAN: Koja | 2019-04
DIKECUALIKAN: Cilincing | 2019-04
2019-04 | Pademangan | Valid: 4,370/14,808 (29.51%)
DIKECUALIKAN: Cengkareng | 2019-04
DIKECUALIKAN: Taman Sari | 2019-04
2019-04 | Kebon Jeruk | Valid: 8,837/19,602 (45.08%)
2019-04 | Kalideres | Valid: 9,692/32,187 (30.11%)
2019-04 | Pal Merah | Valid: 2,605/8,188 (31.81%)
2019-04 | Kembangan | Valid: 22,032/27,702 (79.53%)
DIKECUALIKAN: Setiabudi | 2019-04
2019-04 | Kebayoran Lama | Valid: 15,260/21,188 (72.02%)
DIKECUALIKAN: Cilandak | 2019-04
2019-04 | Kebayoran Baru | Valid: 2,862/14,111 (20.28%)
DIKECUALIKAN: Jagakarsa | 2019-04
2019

c:\Users\hendr\Desktop\OWN\Daily Mine\Skripsi\PROGRAM\UHI-CLASSIFICATION\src\preprocessing\function\_11_model.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Epoch [1/200] Train Loss: 1.1305 Val Loss: 0.9842 Patience: 0/30
Epoch [2/200] Train Loss: 0.9654 Val Loss: 0.9306 Patience: 0/30
Epoch [3/200] Train Loss: 0.9365 Val Loss: 0.9248 Patience: 0/30
Epoch [4/200] Train Loss: 0.9196 Val Loss: 0.9165 Patience: 0/30
Epoch [5/200] Train Loss: 0.8967 Val Loss: 0.9129 Patience: 0/30
Epoch [6/200] Train Loss: 0.8791 Val Loss: 0.9083 Patience: 0/30
Epoch [7/200] Train Loss: 0.8709 Val Loss: 0.9111 Patience: 1/30
Epoch [8/200] Train Loss: 0.8339 Val Loss: 0.8975 Patience: 0/30
Epoch [9/200] Train Loss: 0.8385 Val Loss: 0.8896 Patience: 0/30
Epoch [10/200] Train Loss: 0.8087 Val Loss: 0.8748 Patience: 0/30
Epoch [11/200] Train Loss: 0.7887 Val Loss: 0.8419 Patience: 0/30
Epoch [12/200] Train Loss: 0.7885 Val Loss: 0.8283 Patience: 0/30
Epoch [13/200] Train Loss: 0.7705 Val Loss: 0.8129 Patience: 0/30
Epoch [14/200] Train Loss: 0.7598 Val Loss: 0.7809 Patience: 0/30
Epoch [15/200] Train Loss: 0.7566 Val Loss: 0.7804 Patience: 0/30
Epoch [16/200] Trai

In [2]:
import geopandas as gpd

def get_boundary_file(file_path, subdistrict = "Seluruh kecamatan"):
    boundary = gpd.read_file(file_path)
    boundary = boundary[boundary["WADMPR"] == "Daerah Khusus Ibukota Jakarta"]

    if subdistrict == "Seluruh kecamatan":
        boundary = boundary[boundary['WADMKK'] != 'Administrasi Kepulauan Seribu']
    else:
        boundary = boundary[boundary['WADMKC'] == subdistrict]

    return boundary.reset_index(drop = True)

boundary = get_boundary_file(
    "boundary.gpkg"
)

boundary.to_file(
    "boundary_clean.gpkg",
    driver="GPKG"
)